# Chapter 5: extract loss in 60 brews

Reads 60 synthetic Kestrel brews, works out brewhouse efficiency, splits it into conversion (the mash) and lauter (the runoff) and draws Figure 5.2. Look at which half of the split moves when the mill wears from brew 31.

`conversion_eff_pct` is the lab mash result on the same grist. Lauter is brewhouse efficiency divided by conversion, as in chapter 5.2.2.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

b = pd.read_csv("../data/brewhouse_brews.csv")


def sg_from_plato(p):
    return 1 + p / (258.6 - (p / 258.2) * 227.1)


b["theoretical_kg"] = b.grist_kg * (b.malt_fine_extract_dry_pct - b.fine_coarse_diff_pct) / 100 * (1 - b.malt_moisture_pct / 100)
b["actual_kg"] = b.cast_volume_hl * 0.9982 * b.cast_gravity_p * sg_from_plato(b.cast_gravity_p)
b["brewhouse_eff_pct"] = 100 * b.actual_kg / b.theoretical_kg
b["lauter_eff_pct"] = 100 * b.brewhouse_eff_pct / b.conversion_eff_pct
b[["brew_id", "cast_gravity_p", "theoretical_kg", "actual_kg", "brewhouse_eff_pct", "conversion_eff_pct", "lauter_eff_pct"]].round(2).head()

## One brew at 12.0 degrees Plato (chapter 5.1)

In [ ]:
per_hl = 0.9982 * 12.0 * sg_from_plato(12.0)
theoretical = 5300 * 0.768
print(f"SG at 12.0 P: {sg_from_plato(12.0):.5f}; extract a hL {per_hl:.2f} kg")
print(f"Cast extract {300 * per_hl:,.0f} kg of {theoretical:,.0f} kg theoretical: efficiency {100 * 300 * per_hl / theoretical:.1f} percent, {theoretical - 300 * per_hl:,.0f} kg lost")

## Conversion against lauter: before and after the mill wears

In [ ]:
before, after = b.iloc[:30], b.iloc[50:]
split = pd.DataFrame({
    "Brews 1 to 30": before[["brewhouse_eff_pct", "conversion_eff_pct", "lauter_eff_pct", "mill_roll_gap_mm"]].mean(),
    "Brews 51 to 60": after[["brewhouse_eff_pct", "conversion_eff_pct", "lauter_eff_pct", "mill_roll_gap_mm"]].mean(),
}).round(2)
print(split)
print("Conversion falls as the roll gap opens; lauter holds. The split points at the mill, not the sparge.")

## The drop in money (chapter 5.4 and 5.5)

In [ ]:
MALT_USD_KG = 0.90
annual_malt_kg = 5300 * 104500 / 300
e0 = round(before.brewhouse_eff_pct.mean(), 1) / 100
e1 = round(b.brewhouse_eff_pct.iloc[-1], 1) / 100
one_point_kg = annual_malt_kg * (0.926 / 0.916 - 1)
drift_kg = annual_malt_kg * (e0 / e1 - 1)
print(f"Annual malt, lager equivalent: {annual_malt_kg:,.0f} kg (USD {annual_malt_kg * MALT_USD_KG / 1e6:.2f} million)")
print(f"One point (92.6 to 91.6): {one_point_kg:,.0f} kg of malt, USD {round(one_point_kg) * MALT_USD_KG:,.0f}")
print(f"Target {100 * e0:.1f}, brew 60 at {100 * e1:.1f}: {drift_kg:,.0f} kg of malt a year, USD {round(drift_kg) * MALT_USD_KG:,.0f}")

In [ ]:
eff = b.brewhouse_eff_pct
x = range(1, 61)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x, eff, color="#0b4f45")
ax.axhline(92.6, color="grey", ls="--", lw=1)
ax.fill_between(x[30:], eff[30:], 92.6, where=eff[30:] < 92.6, color="#c27c0e", alpha=0.35, label="Malt cost of the drop below target")
ax.axvline(31, color="grey", ls=":", lw=1)
ax.text(31.5, 93.05, "Mill wear from brew 31")
ax.set_ylim(91.4, 93.2)
ax.set_xlabel("Brew number")
ax.set_ylabel("Brewhouse efficiency (%)")
ax.set_title("Figure 5.2: efficiency over 60 brews (synthetic)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Your own data.** Put your brews in the same columns. If you have no lab mash result, leave `conversion_eff_pct` empty and read brewhouse efficiency only.